# IDR Conservation Analysis

This notebook identifies Intrinsically Disordered Regions (IDRs) in *M. tuberculosis* genes and analyzes their conservation.

**Input:** Directory containing `.pkl` files with `pLDDT` (structure confidence) and `nSNP` (non-synonymous SNP) data.
**Output:**
1.  Excel file with IDR details (`overview_idrs.xlsx`).
2.  3D Scatter plot of IDR properties (`IDR_scatterplot.html`).


In [5]:
!pip install more_itertools
!pip install plotly
!pip install selenium
!pip install bs4
import os
import glob
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from more_itertools import consecutive_groups
import plotly.graph_objs as go
import plotly.io as pio
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
from tqdm import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# --- Numpy Compatibility Patch ---
# The pickle files seem to be created with numpy 2.0+, which uses 'numpy._core'
# This environment has numpy < 2.0. We need to alias the modules.
import sys
import numpy
try:
    import numpy._core.numeric
except ImportError:
    # Ensure numpy._core exists as a module
    try:
        import numpy._core
    except ImportError:
        import types
        sys.modules['numpy._core'] = types.ModuleType('numpy._core')
        
    # Map numpy.core.numeric to numpy._core.numeric
    import numpy.core
    
    if 'numpy.core.numeric' in sys.modules:
        sys.modules['numpy._core.numeric'] = sys.modules['numpy.core.numeric']
    else:
        import numpy.core.numeric
        sys.modules['numpy._core.numeric'] = numpy.core.numeric
        
    # Also patch multiarray and umath just in case
    try:
        import numpy.core.multiarray
        sys.modules['numpy._core.multiarray'] = numpy.core.multiarray
    except ImportError:
        pass
        
    try:
        import numpy.core.umath
        sys.modules['numpy._core.umath'] = numpy.core.umath
    except ImportError:
        pass


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [6]:
# --- Configuration ---

# Path to the directory containing raw gene data pickle files
SOURCE_DATA_PATH = "D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/genetic_variation"

# Output directory for results
OUTPUT_PATH = "D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis"

# Parameters for IDR identification
DISORDER_THRESH = 68.8  # pLDDT score below which residue is considered disordered
THRESH_MARGIN = 0.05    # Margin/Tolerance for extending IDRs
WINDOW_SIZE = 4         # Window size for smoothing/extension
MIN_IDR_LENGTH = 10     # Minimum length to consider an IDR

# Parameters for Conservation Filtering
CONS_THRESH = 1         # nSNP count threshold for "conserved" residues
CONSERVATION_FILTER = 0 # Minimum proportion of conserved residues to keep the IDR

# Construct full paths
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    print(f"Created output directory: {OUTPUT_PATH}")
else:
    print(f"Output directory exists: {OUTPUT_PATH}")


Output directory exists: D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis


In [7]:
def get_idr_indices(plddt_list, nsnp_list=None, disorder_thresh=68.8, thresh_margin=0.05, window=4):
    """
    Identifies stretches of disordered residues based on pLDDT scores.
    
    Args:
        plddt_list (list): List of pLDDT scores per residue.
        nsnp_list (list, optional): List of nSNP counts. Used for plotting if enabled.
        disorder_thresh (float): Threshold for disorder (default 68.8).
        thresh_margin (float): Tolerance for extending the region.
        window (int): Window size for local averaging during extension.
        
    Returns:
        list of lists: Each inner list contains indices of residues in an IDR.
    """
    if nsnp_list is None:
        nsnp_list = [0] * len(plddt_list)

    seed_indices = []
    # Identify seed regions: residues below the disorder threshold
    for index, value in enumerate(plddt_list):
        if value < disorder_thresh:
            seed_indices.append(index)

    indices_to_add = []

    # Try to extend IDRs from seed regions
    for cons_indices in consecutive_groups(seed_indices):
        indices = list(cons_indices)
        if not indices:
            continue
            
        right_extreme_index = max(indices)
        left_extreme_index = min(indices)

        # Extend to the right
        for ext_index in range(right_extreme_index + 1, len(plddt_list)):
            # Calculate median of extended stretch to decide whether to continue
            start_window = max(right_extreme_index + 1, ext_index - window)
            metric = np.median(plddt_list[start_window : ext_index + 1])
            
            if metric < (disorder_thresh + disorder_thresh * thresh_margin):
                indices_to_add.append(ext_index)
            else:
                break

        # Extend to the left
        for ext_index in range(left_extreme_index - 1, -1, -1):
            # Calculate median
            end_window = min(left_extreme_index, ext_index + 1 + window)
            metric = np.median(plddt_list[ext_index : end_window])
            
            if metric < (disorder_thresh + disorder_thresh * thresh_margin):
                indices_to_add.append(ext_index)
            else:
                break

    # Combine original seeds and extended indices
    final_indices = sorted(set(seed_indices + indices_to_add))

    # Group valid indices into continuous stretches
    final_indices_list = [list(g) for g in consecutive_groups(final_indices)]
    
    return final_indices_list


In [8]:
def filter_non_conserved_idrs(idr_indices_list, nsnp_list, cons_thresh, conservation_filter):
    """
    Filters IDRs based on conservation scores and calculates metrics.
    
    Args:
        idr_indices_list (list): List of IDR indices (from get_idr_indices).
        nsnp_list (list): nSNP counts for the gene.
        cons_thresh (float): Threshold for nSNP count to be considered conserved.
        conservation_filter (float): Minimum proportion of conserved residues required.
        
    Returns:
        list of tuples: (idr_indices, metrics_dict)
    """
    filtered_idr_indices_list = []
    nsnp_array = np.array(nsnp_list)
    
    for idr_indices in idr_indices_list:
        if not idr_indices:
            continue
            
        # Metrics for the IDR itself
        idr_nsnps = nsnp_array[idr_indices]
        idr_indices_below_thresh = np.where(idr_nsnps < cons_thresh)[0]
        
        idr_cons_ratio = len(idr_indices_below_thresh) / len(idr_indices)
        idr_median_nsnp = np.median(idr_nsnps)
        idr_mean_nsnp = np.mean(idr_nsnps)

        # Metrics for the rest of the gene (Non-IDR)
        all_indices = set(range(len(nsnp_list)))
        non_idr_indices = list(all_indices - set(idr_indices))
        
        # (Optional: Logic for non-IDR metrics kept for reference/future use)
        # if non_idr_indices:
        #     non_idr_nsnps = nsnp_array[non_idr_indices]
        #     non_idr_indices_below_thresh = np.where(non_idr_nsnps < cons_thresh)[0]
        #     non_idr_cons_ratio = len(non_idr_indices_below_thresh) / len(non_idr_indices)
        #     non_idr_median_nsnp = np.median(non_idr_nsnps)
        #     non_idr_mean_nsnp = np.mean(non_idr_nsnps)

        # Filter
        if idr_cons_ratio > conservation_filter:
            metrics = {
                'cons_prop': idr_cons_ratio, 
                'median_nsnps': idr_median_nsnp, 
                'mean_nsnps': idr_mean_nsnp
            }
            filtered_idr_indices_list.append((idr_indices, metrics))
    
    return filtered_idr_indices_list


In [9]:
def get_mycobrowser_locus(gene_name, driver=None):
    """
    Retrieves the locus tag (e.g., RvXXXX) for a given gene name from Mycobrowser using Selenium.
    Returns early if gene_name already looks like a locus (Rv...).
    """
    # Optimization: Use a local dictionary or file if possible instead of live scraping for every gene
    if gene_name.startswith("Rv") and gene_name[2].isdigit():
        return gene_name

    # If we really need to scrape:
    should_quit = False
    if driver is None:
        try:
            options = webdriver.FirefoxOptions()
            options.add_argument('--headless')
            driver = webdriver.Firefox(options=options)
            should_quit = True
        except Exception as e:
            print(f"Warning: Could not initialize Selenium driver. Skipping lookup for {gene_name}. Error: {e}")
            return gene_name

    try:
        url = 'https://mycobrowser.epfl.ch/'
        driver.get(url)
        
        input_field = driver.find_element(By.ID, 'nav_free_text')
        input_field.clear()
        input_field.send_keys(gene_name)
        
        search_button = driver.find_element(By.ID, 'nav_search_button')
        search_button.click()
        
        wait = WebDriverWait(driver, 5)
        # XPath can be fragile; ensure this matches current site structure
        table = wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/div[1]/div/div[2]/table")))
        table_html = table.get_attribute('outerHTML')
        
        soup = BeautifulSoup(table_html, 'html.parser')
        rows = soup.select('tbody tr')
        
        for row in rows:
            columns = row.find_all('td')
            if len(columns) > 1:
                species = columns[1].text.strip()
                found_name = columns[3].text.strip()
                locus = columns[2].text.strip()
                if species == "M. tuberculosis" and found_name == gene_name:
                    return locus
    except Exception as e:
        # print(f"Lookup failed for {gene_name}: {e}")
        pass
    finally:
        if should_quit:
            driver.quit()
            
    return gene_name  # Fallback to original name


In [10]:
def load_summary_data(filepath):
    """Loads a picking summary file."""
    try:
        df = pd.read_pickle(filepath)
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None


In [11]:
# --- Main Processing ---

# Find all summary files
pkl_files = glob.glob(os.path.join(SOURCE_DATA_PATH, "*_summary.pkl"))
print(f"Found {len(pkl_files)} summary files in {SOURCE_DATA_PATH}")

results_data = {
    'gene_names': [],
    'sequences': [],
    'lengths': [],
    'median_plddt_values': [],
    'median_nsnp_counts': [],
    'mean_nsnp_counts': [],
    'cons_props': [],
    # Additional metadata fields could be populated here if available
    'mycobrowser_links': [],    
}

# Process each file
# Use tqdm for progress bar
for file_path in tqdm(pkl_files, desc="Processing Genes"):
    # Extract gene Name/ID from filename (assuming format "RvXXXX_summary.pkl")
    filename = os.path.basename(file_path)
    gene_name = filename.split('_')[0]
    
    # Load Data
    df = load_summary_data(file_path)
    if df is None:
        continue
        
    # Ensure required columns exist
    if 'plddt' not in df.columns or 'nsnps' not in df.columns:
        # print(f"Missing columns in {filename}, skipping.")
        continue
        
    plddt_scores = df['plddt'].tolist()
    nsnp_counts = df['nsnps'].tolist()
    aa_sequence = df['aa'].tolist() # Assuming 'aa' column exists
    
    # 1. Identify IDRs
    idr_indices_list = get_idr_indices(plddt_scores, nsnp_counts, 
                                     disorder_thresh=DISORDER_THRESH, 
                                     thresh_margin=THRESH_MARGIN, 
                                     window=WINDOW_SIZE)
                                     
    # 2. Filter & Calculate Conservation
    filtered_idrs = filter_non_conserved_idrs(idr_indices_list, nsnp_counts, 
                                            cons_thresh=CONS_THRESH, 
                                            conservation_filter=CONSERVATION_FILTER)
                                            
    # 3. Aggregate Results
    for indices, metrics in filtered_idrs:
        if len(indices) < MIN_IDR_LENGTH:
            continue
            
        # Extract sequence for this IDR
        idr_seq = "".join([str(aa_sequence[i]) for i in indices])
        
        # Calculate pLDDT stats for just this IDR
        idr_plddt = [plddt_scores[i] for i in indices]
        median_plddt = np.median(idr_plddt)
        
        # Append to results
        results_data['gene_names'].append(gene_name)
        results_data['sequences'].append(idr_seq)
        results_data['lengths'].append(len(indices))
        results_data['median_plddt_values'].append(median_plddt)
        results_data['median_nsnp_counts'].append(metrics['median_nsnps'])
        results_data['mean_nsnp_counts'].append(metrics['mean_nsnps'])
        results_data['cons_props'].append(metrics['cons_prop'])
        results_data['mycobrowser_links'].append(f'https://mycobrowser.epfl.ch/genes/{gene_name}')

print(f"Extracted {len(results_data['gene_names'])} IDRs.")


Found 1143 summary files in D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/genetic_variation


Processing Genes: 100%|██████████| 1143/1143 [00:02<00:00, 450.36it/s]

Extracted 1483 IDRs.


In [12]:
# Create DataFrame
df_results = pd.DataFrame(results_data)

# Save to Excel
excel_path = os.path.join(OUTPUT_PATH, "overview_idrs.xlsx")
try:
    df_results.to_excel(excel_path, index=False)
    print(f"Successfully saved Excel to: {excel_path}")
except Exception as e:
    print(f"Error saving Excel: {e}")
    # Fallback to CSV if Excel fails (e.g. missing openpyxl)
    csv_path = excel_path.replace('.xlsx', '.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"Saved as CSV instead: {csv_path}")


Error saving Excel: No module named 'openpyxl'
Saved as CSV instead: D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis\overview_idrs.csv


In [13]:
# --- Visualization ---

def wrap_text(text, width=40):
    """Helper to wrap text for hover tooltips."""
    return "<br>".join([text[i:i+width] for i in range(0, len(text), width)])

# Prepare data for plotting
df_results['sequences_wrapped'] = df_results['sequences'].apply(lambda x: wrap_text(x, 60))

# Create 3D Scatter Plot
fig = go.Figure(data=[go.Scatter3d(
    x=df_results['median_plddt_values'],
    y=df_results['lengths'],
    z=df_results['cons_props'],
    mode='markers',
    marker=dict(
        # Size based on mean nSNPs (inverse or direct scaling as needed)
        # Here assuming user want larger for fewer SNPs? The original code had specific scaling logic:
        # scaled_diameters = [min(5/e, 15) if e > 0 else 25 for e in mean_nsnp_counts]
        size=[min(5/e, 15) if e > 0 else 25 for e in df_results['mean_nsnp_counts']],
        sizemode='diameter',
        color=df_results['median_plddt_values'], # Color by pLDDT for variety
        colorscale='Portland',
        opacity=0.8
    ),
    text=df_results['gene_names'], # Quick label
    customdata=np.stack((
        df_results['gene_names'], 
        df_results['lengths'],
        df_results['sequences_wrapped'],
        df_results['mycobrowser_links']
    ), axis=-1),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>" +
        "Median pLDDT: %{x:.2f}<br>" +
        "Length: %{y}<br>" +
        "Conserved Prop: %{z:.2f}<br>" +
        "Sequence: %{customdata[2]}<br>" +
        "<a href='%{customdata[3]}'>Mycobrowser</a>" +
        "<extra></extra>"
    )
)])

fig.update_layout(
    title="Conserved IDRs in M. tuberculosis",
    scene=dict(
        xaxis=dict(title='Median pLDDT (lower=more disordered)', range=[20, 70]),
        yaxis=dict(title='IDR Length', range=[10, 200]),
        zaxis=dict(title='Conservation Proportion', range=[0, 1.05])
    ),
    margin=dict(l=0, r=0, b=0, t=30)
)

# Save Plot
html_path = os.path.join(OUTPUT_PATH, "IDR_scatterplot.html")
pio.write_html(fig, file=html_path, auto_open=False)
print(f"Saved 3D plot to: {html_path}")

# fig.show() # Uncomment to view in notebook


Saved 3D plot to: D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis\IDR_scatterplot.html


# Motif Search
## Using MEME suite

In [1]:
# ---using MEME suite---
# Export IDR sequences to FASTA format
with open("idr_sequences.fasta", "w") as f:
    for i in range(len(results_data['gene_names'])):
        gene = results_data['gene_names'][i]
        seq = results_data['sequences'][i]
        f.write(f">{gene}_IDR_{i}\n{seq}\n")

NameError: name 'results_data' is not defined